In [2]:
import pandas as pd

# --- Load new dataset ---
new_df = pd.read_csv("Apple_history.csv")   # update filename/path
df = pd.read_csv("Apple_daily.csv")

print("Shape:", new_df.shape)
print("\nColumns:", new_df.columns.tolist())
print("\nDtypes:\n", new_df.dtypes)
print("\nFirst 5 rows:\n", new_df.head())
print("\nLast 5 rows:\n", new_df.tail())

# --- Check what the date column actually looks like before touching it ---
# Try common date column names
possible_date_cols = [c for c in new_df.columns if "date" in c.lower() or "time" in c.lower()]
print("\nLikely date column(s):", possible_date_cols)

for col in possible_date_cols:
    print(f"\n--- Sample raw values from '{col}' ---")
    print(new_df[col].head(10).tolist())
    print(f"Unique value lengths (checks for mixed formats): {new_df[col].astype(str).str.len().unique()}")

Shape: (10068, 6)

Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

Dtypes:
 Date       object
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object

First 5 rows:
                   Date  Open  High   Low  Close  Volume
0  26/09/1986 16:00:00  0.15  0.15  0.15   0.15  312600
1  29/09/1986 16:00:00  0.15  0.15  0.14   0.15  932800
2  30/09/1986 16:00:00  0.15  0.15  0.15   0.15  807100
3  01/10/1986 16:00:00  0.15  0.15  0.15   0.15  618700
4  02/10/1986 16:00:00  0.15  0.15  0.15   0.15  423300

Last 5 rows:
                       Date    Open    High     Low   Close    Volume
10063  04/09/2026 16:00:00  328.31  328.93  317.86  319.97  39606885
10064  08/09/2026 16:00:00  317.10  320.70  314.90  316.22  35477091
10065  09/09/2026 16:00:00  315.49  319.15  309.90  315.34  65639962
10066  10/09/2026 16:00:00  316.67  326.74  316.51  326.57  70011913
10067  11/09/2026 16:00:00  327.45  336.22  326.30  332.27  50716865

Likely 

In [3]:
import pandas as pd
import numpy as np

# --- Step 1: Inspect both sources ---
print("YFINANCE (df)")
print(df.head())
print("Shape:", df.shape)

print("\nGoogle Data (new_df)")
print(new_df.head())
print("Shape:", new_df.shape)

print("\nYFINANCE COLUMNS:", df.columns.tolist())
print("YFINANCE DTYPES:\n", df.dtypes)
print("\nGoogle Data COLUMNS:", new_df.columns.tolist())
print("Google Data DTYPES:\n", new_df.dtypes)

YFINANCE (df)
                        Date      Open      High       Low     Close  \
0  1986-09-22 00:00:00-04:00  0.114334  0.120733  0.114334  0.120307   
1  1986-09-23 00:00:00-04:00  0.120307  0.123719  0.119880  0.123293   
2  1986-09-24 00:00:00-04:00  0.123293  0.124146  0.116041  0.119880   
3  1986-09-25 00:00:00-04:00  0.119880  0.120307  0.114761  0.117747   
4  1986-09-26 00:00:00-04:00  0.116467  0.117321  0.115614  0.116894   

      Volume  Dividends  Stock Splits  
0  239680000        0.0           0.0  
1  338240000        0.0           0.0  
2  176870400        0.0           0.0  
3  187801600        0.0           0.0  
4   70022400        0.0           0.0  
Shape: (10070, 8)

Google Data (new_df)
                  Date  Open  High   Low  Close  Volume
0  26/09/1986 16:00:00  0.15  0.15  0.15   0.15  312600
1  29/09/1986 16:00:00  0.15  0.15  0.14   0.15  932800
2  30/09/1986 16:00:00  0.15  0.15  0.15   0.15  807100
3  01/10/1986 16:00:00  0.15  0.15  0.15   0.15  

In [4]:
print(df["Date"].dtype)
print(df["Date"].head(10))
print(df["Date"].apply(type).value_counts())

object
0    1986-09-22 00:00:00-04:00
1    1986-09-23 00:00:00-04:00
2    1986-09-24 00:00:00-04:00
3    1986-09-25 00:00:00-04:00
4    1986-09-26 00:00:00-04:00
5    1986-09-29 00:00:00-04:00
6    1986-09-30 00:00:00-04:00
7    1986-10-01 00:00:00-04:00
8    1986-10-02 00:00:00-04:00
9    1986-10-03 00:00:00-04:00
Name: Date, dtype: object
Date
<class 'str'>    10070
Name: count, dtype: int64


In [5]:
df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None)
df["Date"] = df["Date"].dt.normalize()

print(df["Date"].dtype)
print(df["Date"].head())

datetime64[ns]
0   1986-09-22
1   1986-09-23
2   1986-09-24
3   1986-09-25
4   1986-09-26
Name: Date, dtype: datetime64[ns]


In [6]:
# --- Step 2: Normalize dates on both sides (strip time, sort) ---
df["Date"] = pd.to_datetime(df["Date"])
new_df["Date"] = pd.to_datetime(new_df["Date"], format="%d/%m/%Y %H:%M:%S")

df["Date"] = df["Date"].dt.normalize()
new_df["Date"] = new_df["Date"].dt.normalize()

df = df.sort_values("Date").reset_index(drop=True)
new_df = new_df.sort_values("Date").reset_index(drop=True)

print("YFINANCE")
print("Records:", len(df))
print("Date range:", df["Date"].min(), "to", df["Date"].max())

print("\nGoogle Data")
print("Records:", len(new_df))
print("Date range:", new_df["Date"].min(), "to", new_df["Date"].max())

YFINANCE
Records: 10070
Date range: 1986-09-22 00:00:00 to 2026-09-11 00:00:00

Google Data
Records: 10068
Date range: 1986-09-26 00:00:00 to 2026-09-11 00:00:00


In [7]:
# --- Step 3: Compare date coverage between sources ---
yf_dates = set(df["Date"])
new_dates = set(new_df["Date"])

matching_dates = yf_dates & new_dates
yf_only = yf_dates - new_dates
new_only = new_dates - yf_dates

print("DATE COMPARISON")
print(f"yfinance records   : {len(df)}")
print(f"Google Data records : {len(new_df)}")
print(f"Matching dates     : {len(matching_dates)}")
print(f"yfinance only dates: {len(yf_only)}")
print(f"Google Data only    : {len(new_only)}")

if yf_only:
    print("\nDates only in yfinance:")
    print(sorted(yf_only)[:10], "..." if len(yf_only) > 10 else "")

if new_only:
    print("\nDates only in Google Data:")
    print(sorted(new_only)[:10], "..." if len(new_only) > 10 else "")

DATE COMPARISON
yfinance records   : 10070
Google Data records : 10068
Matching dates     : 10066
yfinance only dates: 4
Google Data only    : 2

Dates only in yfinance:
[Timestamp('1986-09-22 00:00:00'), Timestamp('1986-09-23 00:00:00'), Timestamp('1986-09-24 00:00:00'), Timestamp('1986-09-25 00:00:00')] 

Dates only in Google Data:
[Timestamp('1995-04-14 00:00:00'), Timestamp('1996-11-28 00:00:00')] 


In [8]:
# --- Step 4: Merge on common dates, compare actual price values ---
comparison = pd.merge(
    df[["Date", "Open", "High", "Low", "Close", "Volume"]],
    new_df[["Date", "Open", "High", "Low", "Close", "Volume"]],
    on="Date", suffixes=("_yfinance", "_new")
)

print("Common records:", len(comparison))
print(comparison.head())

# --- Step 5: Quantify agreement -- % difference per column ---
for col in ["Open", "High", "Low", "Close"]:
    diff_pct = ((comparison[f"{col}_new"] - comparison[f"{col}_yfinance"]) / comparison[f"{col}_yfinance"]) * 100
    print(f"\n{col} difference (%): mean={diff_pct.mean():.4f}, "
          f"max abs={diff_pct.abs().max():.4f}, "
          f"rows with >1% diff={(diff_pct.abs() > 1).sum()}")

Common records: 10066
        Date  Open_yfinance  High_yfinance  Low_yfinance  Close_yfinance  \
0 1986-09-26       0.116467       0.117321      0.115614        0.116894   
1 1986-09-29       0.114761       0.115614      0.107935        0.110921   
2 1986-09-30       0.112201       0.115614      0.111348        0.114334   
3 1986-10-01       0.113908       0.117747      0.113908        0.116467   
4 1986-10-02       0.115188       0.117321      0.114334        0.116467   

   Volume_yfinance  Open_new  High_new  Low_new  Close_new  Volume_new  
0         70022400      0.15      0.15     0.15       0.15      312600  
1        208947200      0.15      0.15     0.14       0.15      932800  
2        180790400      0.15      0.15     0.15       0.15      807100  
3        138588800      0.15      0.15     0.15       0.15      618700  
4         94819200      0.15      0.15     0.15       0.15      423300  

Open difference (%): mean=16.3495, max abs=44.2465, rows with >1% diff=9480

High 

In [9]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [10]:
import plotly.graph_objects as go

metrics = ["Close", "Open", "High", "Low"]
sources = {"yfinance": "#636efa", "New Source": "#EF553B"}  # plotly's default blue/red
colors = {"yfinance": "Close_yfinance", "New Source": "Close_new"}

def col_name(metric, source):
    suffix = "yfinance" if source == "yfinance" else "new"
    return f"{metric}_{suffix}"

fig = go.Figure()
trace_index = {}  # maps metric -> [trace positions] for the dropdown logic

pos = 0
for metric in metrics:
    trace_index[metric] = []
    for source, color in sources.items():
        fig.add_trace(go.Scatter(
            x=comparison["Date"],
            y=comparison[col_name(metric, source)],
            mode="lines",
            name=source,
            line=dict(color=color, width=1.3),
            visible=(metric == "Close"),
            hovertemplate=f"%{{x|%d %b %Y}}<br>{source} {metric}: $%{{y:.2f}}<extra></extra>"
        ))
        trace_index[metric].append(pos)
        pos += 1

total_traces = pos

def visibility_mask(active_metric):
    return [i in trace_index[active_metric] for i in range(total_traces)]

dropdown = dict(
    buttons=[
        dict(
            label=m,
            method="update",
            args=[
                {"visible": visibility_mask(m)},
                {"title": f"AAPL {m} Price Comparison", "yaxis": {"title": f"{m} Price (USD)"}}
            ]
        ) for m in metrics
    ],
    direction="down",
    x=1.0, xanchor="right",
    y=1.18, yanchor="top",
    bgcolor="#f5f5f5",
    bordercolor="#999",
    borderwidth=1
)

fig.update_layout(
    title=dict(text="AAPL Close Price Comparison — yfinance vs New Source", x=0.02, xanchor="left"),
    xaxis_title="Date",
    yaxis_title="Close Price (USD)",
    hovermode="x unified",
    template="simple_white",
    height=650,
    updatemenus=[dropdown],
    legend=dict(orientation="v", yanchor="top", y=0.98, xanchor="left", x=0.01),
    margin=dict(l=70, r=30, t=90, b=60)
)

fig.show()

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# --- 1. Basic descriptive statistics ---
print("=== Descriptive Statistics ===")
print(df[["Close", "Volume", "Daily_Return"]].describe())

print(f"\nSkewness of Daily_Return: {df['Daily_Return'].skew():.4f}")
print(f"Kurtosis of Daily_Return: {df['Daily_Return'].kurtosis():.4f}")

# --- 2. Price trend over 40 years (log scale, since price spans 0.11 to 332) ---
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(df.index, df["Close"], color="#34A853", linewidth=0.7)
axes[0].set_title("AAPL Close Price — Linear Scale (1986–2026)")
axes[0].set_ylabel("Price (USD)")

axes[1].plot(df.index, df["Close"], color="#4285F4", linewidth=0.7)
axes[1].set_yscale("log")
axes[1].set_title("AAPL Close Price — Log Scale (reveals early-era detail)")
axes[1].set_ylabel("Price (USD, log)")

plt.tight_layout()
plt.show()

# --- 3. Volume trend over time ---
plt.figure(figsize=(14, 4))
plt.plot(df.index, df["Volume"], color="gray", linewidth=0.5)
plt.title("AAPL Trading Volume Over Time")
plt.ylabel("Volume")
plt.tight_layout()
plt.show()

# --- 4. Daily return distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["Daily_Return"].dropna(), bins=150, color="#4285F4", edgecolor="none")
axes[0].set_title("Distribution of Daily Returns")
axes[0].set_xlabel("Daily Return")
axes[0].axvline(0, color="red", linestyle="--", linewidth=1)

sns.boxplot(x=df["Daily_Return"].dropna(), ax=axes[1], color="#34A853")
axes[1].set_title("Daily Return — Boxplot (outlier view)")

plt.tight_layout()
plt.show()

# --- 5. Rolling volatility over time (volatility clustering) ---
plt.figure(figsize=(14, 4))
plt.plot(df.index, df["Volatility_30"], color="#EA4335", linewidth=0.7)
plt.title("30-Day Rolling Volatility Over Time (Volatility Clustering)")
plt.ylabel("Rolling Std of Daily Return")
plt.tight_layout()
plt.show()

# --- 6. Stationarity test (ADF) — Close price vs Daily Return ---
def run_adf(series, label):
    result = adfuller(series.dropna())
    print(f"\nADF Test — {label}")
    print(f"  Statistic: {result[0]:.4f}")
    print(f"  p-value:   {result[1]:.4f}")
    print(f"  Stationary: {'YES' if result[1] < 0.05 else 'NO'}")

run_adf(df["Close"], "Close Price (raw)")
run_adf(df["Daily_Return"], "Daily Return")

# --- 7. ACF / PACF plots (autocorrelation structure) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_acf(df["Close"].dropna(), lags=40, ax=axes[0, 0])
axes[0, 0].set_title("ACF — Close Price")
plot_pacf(df["Close"].dropna(), lags=40, ax=axes[0, 1])
axes[0, 1].set_title("PACF — Close Price")
plot_acf(df["Daily_Return"].dropna(), lags=40, ax=axes[1, 0])
axes[1, 0].set_title("ACF — Daily Return")
plot_pacf(df["Daily_Return"].dropna(), lags=40, ax=axes[1, 1])
axes[1, 1].set_title("PACF — Daily Return")
plt.tight_layout()
plt.show()

# --- 8. Seasonality: average return by month and by day-of-week ---
df_season = df.copy()
df_season["Month"] = df_season.index.month
df_season["DayOfWeek"] = df_season.index.day_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

monthly_avg = df_season.groupby("Month")["Daily_Return"].mean()
axes[0].bar(monthly_avg.index, monthly_avg.values, color="#4285F4")
axes[0].set_title("Average Daily Return by Month")
axes[0].set_xlabel("Month")

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
dow_avg = df_season.groupby("DayOfWeek")["Daily_Return"].mean().reindex(dow_order)
axes[1].bar(dow_avg.index, dow_avg.values, color="#34A853")
axes[1].set_title("Average Daily Return by Day of Week")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# --- 9. Yearly volatility comparison (which years were riskiest) ---
df_season["Year"] = df_season.index.year
yearly_vol = df_season.groupby("Year")["Daily_Return"].std()

plt.figure(figsize=(14, 4))
plt.bar(yearly_vol.index, yearly_vol.values, color="#EA4335")
plt.title("Yearly Volatility (Std of Daily Returns) — AAPL 1986–2026")
plt.ylabel("Std Dev of Daily Return")
plt.tight_layout()
plt.show()

=== Descriptive Statistics ===


KeyError: "['Daily_Return'] not in index"

In [13]:
print("Current columns in df:", df.columns.tolist())
print("Current columns in df:", new_df.columns.tolist())

Current columns in df: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']
Current columns in df: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# --- Safety check: ensure Daily_Return exists ---
if "Daily_Return" not in df.columns:
    df["Daily_Return"] = df["Close"].pct_change()

if "Volatility_30" not in df.columns:
    df["Volatility_30"] = df["Daily_Return"].rolling(window=30).std()

# --- 1. Basic descriptive statistics ---
print("=== Descriptive Statistics ===")
print(df[["Close", "Volume", "Daily_Return"]].describe())

print(f"\nSkewness of Daily_Return: {df['Daily_Return'].skew():.4f}")
print(f"Kurtosis of Daily_Return: {df['Daily_Return'].kurtosis():.4f}")

=== Descriptive Statistics ===
              Close        Volume  Daily_Return
count  10070.000000  1.007000e+04  10069.000000
mean      36.374435  3.300738e+08      0.001149
std       68.757754  3.452753e+08      0.026667
min        0.096739  2.396800e+06     -0.518692
25%        0.296124  1.130206e+08     -0.011630
50%        2.234249  2.141384e+08      0.000443
75%       27.754766  4.184803e+08      0.013748
max      339.786926  7.421641e+09      0.332281

Skewness of Daily_Return: -0.3961
Kurtosis of Daily_Return: 23.0994
